# 01 — Implantation de Citiz et de la concurrence en autopartage

**Objectif** : cartographier les stations Citiz en France, superposer les concurrents en autopartage et poser une première lecture des **parts de marché** (en nombre de véhicules) par département.

**Sources (open data, flux GBFS, transport.data.gouv.fr)**

| Opérateur | Modèle | Flux |
|---|---|---|
| Citiz (14 réseaux régionaux + Citiz Lyon) | Autopartage en station (boucle) | `backend.citiz.fr/public/provider/<id>/gbfs` + `data.grandlyon.com` |
| Getaround | Location entre particuliers + flotte connectée, véhicules garés librement | `fr.getaround.com/gbfs/manifest?country_code=FR` |
| Clem' | Autopartage électrique en station | `gbfs.clem.mobi` |
| Rémi autopartage (Centre-Val de Loire, opéré par Clem') | Autopartage en station | `gbfs.clem.mobi/remi` |
| Optymo (Belfort) | Autopartage en station | `data.optymo.fr/als` |
| Leo&Go (Lyon) | Autopartage en free-floating | `download.data.grandlyon.com` |
| Contours des départements (y c. DROM) | — | `github.com/gregoiredavid/france-geojson` (dérivé IGN Admin Express) |

⚠️ **Limites à garder en tête** (détaillées en fin de notebook) : photo à un instant T, opérateurs sans flux GBFS absents (Communauto, Zity, Free2Move, Ubeeqo, Marcel…), et Getaround n'est pas un concurrent direct au même titre (véhicules de particuliers).

In [ ]:
# Dépendances : pip install pandas requests folium shapely matplotlib
import json, math, time, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
import folium
from folium.plugins import FastMarkerCluster
import shapely
from shapely.geometry import shape
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

DATA_DIR = Path("../data/raw"); DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("../outputs"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# Passer à True pour re-télécharger les flux (sinon on réutilise le cache local du jour)
FORCE_REFRESH = False

## 1. Outils de lecture GBFS

Les flux sont en GBFS 2.3 ou 3.0 : les deux versions n'ont pas tout à fait la même structure (langue, noms des fichiers). Les fonctions ci-dessous lissent ces différences.

In [ ]:
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "hackathon-mobility-citiz/0.1 (projet etudiant Epitech)"

def fetch_json(url, retries=3):
    for i in range(retries):
        try:
            r = SESSION.get(url, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception:
            if i == retries - 1:
                raise
            time.sleep(1 + i)

def gbfs_feeds(gbfs_url):
    # Renvoie {nom_du_flux: url} pour GBFS 2.x (data.<langue>.feeds) et 3.x (data.feeds)
    data = fetch_json(gbfs_url)["data"]
    feeds = data["feeds"] if "feeds" in data else data[next(iter(data))]["feeds"]
    return {f["name"]: f["url"] for f in feeds}

def txt(x):
    # GBFS 3.0 : les libellés sont des listes [{"text":..., "language":...}]
    if isinstance(x, list):
        return x[0].get("text", "").strip() if x else ""
    return (x or "").strip() if isinstance(x, str) else x

def load_stations(gbfs_url, operator, network):
    # Stations + nombre de véhicules rattachés à chaque station
    feeds = gbfs_feeds(gbfs_url)
    info = pd.DataFrame(fetch_json(feeds["station_information"])["data"]["stations"])
    info["station_id"] = info["station_id"].astype(str)
    df = pd.DataFrame({
        "operator": operator,
        "network": network,
        "id": info["station_id"],
        "name": info["name"].map(txt) if "name" in info else "",
        "lat": info["lat"].astype(float),
        "lon": info["lon"].astype(float),
        "capacity": info["capacity"] if "capacity" in info else np.nan,
    })

    # 1) Meilleure source : vehicle_status, compté par station d'attache (inclut les voitures en location)
    n_veh = None
    if "vehicle_status" in feeds:
        veh = pd.DataFrame(fetch_json(feeds["vehicle_status"])["data"].get("vehicles", []))
        if len(veh):
            key = "home_station_id" if "home_station_id" in veh else "station_id"
            n_veh = veh[key].astype(str).value_counts()
    if n_veh is not None:
        df["n_vehicles"] = df["id"].map(n_veh).fillna(0)
        df["vehicles_source"] = "vehicle_status"
    else:
        # 2) Sinon : véhicules disponibles (station_status), à défaut la capacité de la station
        status = pd.DataFrame(fetch_json(feeds["station_status"])["data"]["stations"])
        status["station_id"] = status["station_id"].astype(str)
        avail = status.set_index("station_id").get("num_vehicles_available",
                status.set_index("station_id").get("num_bikes_available"))
        df["n_vehicles"] = np.fmax(df["id"].map(avail).astype(float), df["capacity"].astype(float))
        df["vehicles_source"] = "station_status/capacity"
    df["model"] = "station"
    return df

## 2. Collecte — Citiz

Citiz est un réseau coopératif : chaque région a son propre flux. On les agrège tous.

In [ ]:
CITIZ_FEEDS = {
    "Citiz Grand Est":            "https://backend.citiz.fr/public/provider/1/gbfs/v3.0/gbfs.json",
    "Citiz Bourgogne-Franche-Comté": "https://backend.citiz.fr/public/provider/2/gbfs/3.0/gbfs.json",
    "Citiz Gironde":              "https://backend.citiz.fr/public/provider/4/gbfs/v3.0/gbfs.json",
    "Citiz Auvergne-Rhône-Alpes": "https://backend.citiz.fr/public/provider/5/gbfs/3.0/gbfs.json",
    "Citiz Provence":             "https://backend.citiz.fr/public/provider/6/gbfs/v3.0/gbfs.json",
    "Citiz Angers":               "https://backend.citiz.fr/public/provider/7/gbfs/v3.0/gbfs.json",
    "Citiz Occitanie":            "https://backend.citiz.fr/public/provider/8/gbfs/3.0/gbfs.json",
    "Citiz Grand Poitiers":       "https://backend.citiz.fr/public/provider/9/gbfs/3.0/gbfs.json",
    "Citiz Développement":        "https://backend.citiz.fr/public/provider/12/gbfs/3.0/gbfs.json",
    "Citiz Nantes":               "https://backend.citiz.fr/public/provider/15/gbfs/v3.0/gbfs.json",
    "Citiz Hauts-de-France":      "https://backend.citiz.fr/public/provider/16/gbfs/v3.0/gbfs.json",
    "Citiz La Rochelle":          "https://backend.citiz.fr/public/provider/18/gbfs/v3.0/gbfs.json",
    "Citiz Rennes":               "https://backend.citiz.fr/public/provider/19/gbfs/v3.0/gbfs.json",
    "Citiz Pays Basque":          "https://backend.citiz.fr/public/provider/20/gbfs/3.0/gbfs.json",
    "Citiz Lyon":                 "https://data.grandlyon.com/files/rdata/lpa_mobilite.donnees/gbfs.json",
}

today = pd.Timestamp.now().strftime("%Y-%m-%d")
cache = DATA_DIR / f"citiz_stations_{today}.csv"
if cache.exists() and not FORCE_REFRESH:
    citiz = pd.read_csv(cache, dtype={"id": str})
else:
    parts = []
    for network, url in CITIZ_FEEDS.items():
        try:
            parts.append(load_stations(url, "Citiz", network))
        except Exception as e:
            print(f"⚠️ {network} : {e}")
    citiz = pd.concat(parts, ignore_index=True)
    citiz.to_csv(cache, index=False)

# Un même lieu peut apparaître dans deux flux (ex. Lyon) : dédoublonnage sur les coordonnées arrondies (~10 m)
before = len(citiz)
citiz = (citiz.assign(_k=citiz["lat"].round(4).astype(str) + "," + citiz["lon"].round(4).astype(str))
              .sort_values("n_vehicles", ascending=False)
              .drop_duplicates("_k").drop(columns="_k"))
print(f"{len(citiz)} stations Citiz ({before - len(citiz)} doublons retirés), "
      f"{int(citiz['n_vehicles'].sum())} véhicules")
citiz.groupby("network").agg(stations=("id", "size"), vehicules=("n_vehicles", "sum"),
                             source=("vehicles_source", "first")).sort_values("stations", ascending=False)

## 3. Collecte — concurrents en autopartage

- **Clem', Rémi, Optymo** : stations, comme Citiz.
- **Leo&Go** : véhicules en free-floating (un point = une voiture).
- **Getaround** : environ 3 700 flux (un par ville). Ils se recoupent, donc on dédoublonne par `vehicle_id`. Le serveur limite le débit, donc la collecte est lente (~45 min) et passe par `scripts/collect_getaround.py`, avec cache et reprise.

In [ ]:
def load_free_vehicles(gbfs_url, operator, network):
    feeds = gbfs_feeds(gbfs_url)
    name = "vehicle_status" if "vehicle_status" in feeds else "free_bike_status"
    data = fetch_json(feeds[name])["data"]
    v = pd.DataFrame(data.get("vehicles", data.get("bikes", [])))
    vid = "vehicle_id" if "vehicle_id" in v else "bike_id"
    return pd.DataFrame({"operator": operator, "network": network, "id": v[vid].astype(str),
                         "name": "", "lat": v["lat"].astype(float), "lon": v["lon"].astype(float),
                         "capacity": np.nan, "n_vehicles": 1.0,
                         "vehicles_source": name, "model": "free-floating"})

STATION_COMPETITORS = {
    ("Clem'", "Clem' national"):     "https://gbfs.clem.mobi/gbfs.json",
    ("Clem'", "Rémi Centre-Val de Loire"): "https://gbfs.clem.mobi/remi/gbfs.json",
    ("Optymo", "Optymo Belfort"):    "https://data.optymo.fr/als/gbfs.json",
}
FREE_COMPETITORS = {
    ("Leo&Go", "Leo&Go Lyon"): "https://download.data.grandlyon.com/files/rdata/lag_leoandgo.disponibilite/gbfs.json",
}

cache = DATA_DIR / f"competitors_{today}.csv"
if cache.exists() and not FORCE_REFRESH:
    others = pd.read_csv(cache, dtype={"id": str})
else:
    parts = []
    for (op, network), url in STATION_COMPETITORS.items():
        try: parts.append(load_stations(url, op, network))
        except Exception as e: print(f"⚠️ {network} : {e}")
    for (op, network), url in FREE_COMPETITORS.items():
        try: parts.append(load_free_vehicles(url, op, network))
        except Exception as e: print(f"⚠️ {network} : {e}")
    others = pd.concat(parts, ignore_index=True)
    others.to_csv(cache, index=False)

others.groupby(["operator", "network", "model"]).agg(points=("id", "size"), vehicules=("n_vehicles", "sum"))

In [ ]:
# Getaround limite le débit (erreur 429) : la collecte se fait avec scripts/collect_getaround.py,
# qui va lentement, reprend là où il s'est arrêté et met chaque flux en cache.
# Le plus simple est de le lancer à part une fois (~45 min) : python3 scripts/collect_getaround.py
import sys
sys.path.insert(0, "../scripts")
from collect_getaround import collect

files = sorted(DATA_DIR.glob("getaround_vehicles_*.csv"))
complete = [f for f in files if not f.stem.endswith("_partiel")]
if complete and not FORCE_REFRESH:
    src = complete[-1]
    getaround = pd.read_csv(src, dtype={"id": str})
    print(f"Cache complet : {src.name}")
else:
    # Collecte (ou reprise) limitée à 5 min dans le notebook ; le résultat peut être partiel
    getaround, n_done, n_total = collect(max_minutes=5)
    print(f"⚠️ Couverture Getaround : {n_done}/{n_total} flux ({n_done / n_total:.0%})")

getaround = getaround.assign(id=getaround["id"].astype(str), operator="Getaround", network="Getaround",
                             name="", n_vehicles=1.0, model="particuliers / free")
print(f"{len(getaround):,} véhicules Getaround uniques")

## 4. Table unifiée et rattachement aux départements

In [ ]:
COLS = ["operator", "network", "model", "id", "name", "lat", "lon", "n_vehicles"]
all_points = pd.concat([citiz[COLS], others[COLS], getaround[COLS]], ignore_index=True)
all_points = all_points.dropna(subset=["lat", "lon"])

# Contours des départements (métropole + DROM)
dep_file = DATA_DIR / "departements-avec-outre-mer.geojson"
if not dep_file.exists():
    r = SESSION.get("https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/"
                    "departements-avec-outre-mer.geojson", timeout=60)
    dep_file.write_bytes(r.content)
dep_geo = json.loads(dep_file.read_text())
dep_codes = [f["properties"]["code"] for f in dep_geo["features"]]
dep_names = {f["properties"]["code"]: f["properties"]["nom"] for f in dep_geo["features"]}
dep_shapes = [shape(f["geometry"]) for f in dep_geo["features"]]

tree = shapely.STRtree(dep_shapes)
pts = shapely.points(all_points["lon"].values, all_points["lat"].values)
pt_idx, dep_idx = tree.query(pts, predicate="within")
all_points["dep"] = None
all_points.iloc[pt_idx, all_points.columns.get_loc("dep")] = np.array(dep_codes, dtype=object)[dep_idx]
all_points["dep_nom"] = all_points["dep"].map(dep_names)

hors_france = all_points["dep"].isna().sum()
print(f"{len(all_points):,} points, dont {hors_france} hors France (frontaliers, coordonnées erronées) écartés")
all_points = all_points.dropna(subset=["dep"])
all_points.to_csv(OUT_DIR / f"autopartage_points_{today}.csv", index=False)

## 5. Carte de France — Citiz vs concurrents

- **Citiz** : un cercle par station, dont la taille dépend du nombre de véhicules.
- **Concurrents** : une couleur par opérateur. Getaround est regroupé en clusters (plusieurs dizaines de milliers de points).
- Les couches s'activent ou se désactivent en haut à droite. La carte est aussi exportée en HTML dans `outputs/`.

In [ ]:
COLORS = {
    "Citiz": "#00843D",      # vert
    "Getaround": "#7B2CBF",  # violet
    "Clem'": "#F77F00",      # orange
    "Optymo": "#D62828",     # rouge
    "Leo&Go": "#0077B6",     # bleu
}

m = folium.Map(location=[46.6, 2.4], zoom_start=6, tiles=None, control_scale=True)
folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}", attr="Tiles © Esri — Esri, DeLorme, NAVTEQ", name="Fond clair (Esri)").add_to(m)

# Getaround d'abord (en dessous), en clusters pour rester fluide
g = all_points[all_points["operator"] == "Getaround"]
callback = f'''function (row) {{
    return L.circleMarker(new L.LatLng(row[0], row[1]),
        {{radius: 3, color: "{COLORS['Getaround']}", fillOpacity: 0.6, weight: 0}});
}}'''
fg = folium.FeatureGroup(name=f"Getaround ({len(g):,} véhicules)", show=False)
FastMarkerCluster(g[["lat", "lon"]].values.tolist(), callback=callback).add_to(fg)
fg.add_to(m)

# Autres concurrents
for op in ["Clem'", "Optymo", "Leo&Go"]:
    d = all_points[all_points["operator"] == op]
    fg = folium.FeatureGroup(name=f"{op} ({int(d['n_vehicles'].sum())} véhicules)")
    for r in d.itertuples():
        folium.CircleMarker([r.lat, r.lon], radius=3 + math.sqrt(r.n_vehicles),
                            color=COLORS[op], fill=True, fill_opacity=0.7, weight=1,
                            tooltip=f"{r.network} — {r.name} ({int(r.n_vehicles)} véh.)").add_to(fg)
    fg.add_to(m)

# Citiz au-dessus
d = all_points[all_points["operator"] == "Citiz"]
fg = folium.FeatureGroup(name=f"Citiz ({len(d)} stations, {int(d['n_vehicles'].sum())} véhicules)")
for r in d.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=3 + 1.5 * math.sqrt(r.n_vehicles),
                        color=COLORS["Citiz"], fill=True, fill_opacity=0.8, weight=1,
                        tooltip=f"<b>{r.network}</b><br>{r.name}<br>{int(r.n_vehicles)} véhicule(s)").add_to(fg)
fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
legend = "".join(f'<div><span style="display:inline-block;width:12px;height:12px;border-radius:6px;'
                 f'background:{c};margin-right:6px"></span>{op}</div>' for op, c in COLORS.items())
m.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;'
    f'border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3);font:13px sans-serif">{legend}</div>'))

m.save(str(OUT_DIR / "carte_citiz_concurrents.html"))
m

## 6. Parts de marché

On mesure les parts de marché en **nombre de véhicules**, c'est l'indicateur le plus proche de l'offre réelle. Deux périmètres sont présentés :

1. **Tout l'autopartage visible en open data**, Getaround compris.
2. **Autopartage « professionnel »** (flotte détenue par l'opérateur), **hors Getaround**. C'est le vrai périmètre concurrentiel de Citiz, parce que Getaround est surtout une place de marché entre particuliers.

In [ ]:
def share_table(df):
    t = df.groupby("operator").agg(points=("id", "size"), vehicules=("n_vehicles", "sum"),
                                   departements=("dep", "nunique"))
    t["part_vehicules_%"] = 100 * t["vehicules"] / t["vehicules"].sum()
    return t.sort_values("vehicules", ascending=False)

pro = all_points[all_points["operator"] != "Getaround"]
print("— Tout l'autopartage (open data) —")
display(share_table(all_points))
print("— Autopartage professionnel (hors Getaround) —")
display(share_table(pro))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for ax, (title, df) in zip(axes, [("Tout l'autopartage", all_points), ("Hors Getaround", pro)]):
    t = share_table(df).iloc[::-1]
    ax.barh(t.index, t["part_vehicules_%"], color=[COLORS[o] for o in t.index])
    for y, (p, v) in enumerate(zip(t["part_vehicules_%"], t["vehicules"])):
        ax.text(p + 1, y, f"{p:.1f} %  ({int(v):,} véh.)", va="center", fontsize=9)
    ax.set_title(f"{title} — {int(t['vehicules'].sum()):,} véhicules")
    ax.set_xlim(0, 110); ax.set_xlabel("Part des véhicules (%)")
    ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("Parts de marché nationales en nombre de véhicules", fontweight="bold")
plt.tight_layout(); plt.show()

### 6.1 Par département

Pour chaque département : véhicules par opérateur, part de Citiz sur les deux périmètres et un statut :
- **Citiz leader** : Citiz a plus de 50 % des véhicules professionnels.
- **Citiz challengé** : Citiz est présent mais minoritaire.
- **Citiz seul** : aucun concurrent professionnel visible.
- **Absent, concurrence pro** : un concurrent professionnel est là, Citiz non. C'est une piste de conquête.
- **Absent, Getaround seul** : il existe une demande entre particuliers, sans offre professionnelle.

In [ ]:
pivot = all_points.pivot_table(index=["dep", "dep_nom"], columns="operator", values="n_vehicles",
                               aggfunc="sum", fill_value=0)
for op in COLORS:
    if op not in pivot: pivot[op] = 0
pivot["total"] = pivot[list(COLORS)].sum(axis=1)
pivot["total_pro"] = pivot["total"] - pivot["Getaround"]
pivot["part_citiz_%"] = 100 * pivot["Citiz"] / pivot["total"]
pivot["part_citiz_pro_%"] = 100 * pivot["Citiz"] / pivot["total_pro"].replace(0, np.nan)

def statut(r):
    if r["Citiz"] > 0:
        if r["total_pro"] == r["Citiz"]: return "Citiz seul"
        return "Citiz leader" if r["part_citiz_pro_%"] > 50 else "Citiz challengé"
    return "Absent, concurrence pro" if r["total_pro"] > 0 else "Absent, Getaround seul"

pivot["statut"] = pivot.apply(statut, axis=1)
pivot = pivot.sort_values("total", ascending=False)
pivot.to_csv(OUT_DIR / f"parts_de_marche_departements_{today}.csv")

print(pivot["statut"].value_counts().to_string(), "\n")
pivot[list(COLORS) + ["total", "part_citiz_%", "part_citiz_pro_%", "statut"]].head(30)

In [ ]:
# Parts de marché (100 %) dans les 25 départements où l'offre visible est la plus importante
top = pivot.head(25).iloc[::-1]
labels = [f"{n} ({c}) — {int(t):,} véh." for (c, n), t in zip(top.index, top["total"])]
fig, ax = plt.subplots(figsize=(11, 9))
left = np.zeros(len(top))
for op in ["Citiz", "Clem'", "Optymo", "Leo&Go", "Getaround"]:
    share = 100 * top[op].values / top["total"].values
    ax.barh(labels, share, left=left, color=COLORS[op], label=op)
    left += share
ax.set_xlim(0, 100); ax.set_xlabel("Part des véhicules (%)")
ax.set_title("Parts de marché dans les 25 départements les mieux dotés", fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=5, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# Départements où Citiz est le plus implanté : comparaison avec la concurrence
citiz_deps = pivot[pivot["Citiz"] > 0].sort_values("Citiz", ascending=False)
citiz_deps[["Citiz", "total_pro", "Getaround", "part_citiz_pro_%", "part_citiz_%", "statut"]].head(25)

### 6.2 Carte de la part de Citiz par département (périmètre professionnel)

In [ ]:
# Géométries simplifiées pour alléger la carte
light = {"type": "FeatureCollection", "features": []}
p = pivot.reset_index().set_index("dep")
for f, geom in zip(dep_geo["features"], dep_shapes):
    code_dep = f["properties"]["code"]
    props = {"code": code_dep, "nom": f["properties"]["nom"]}
    if code_dep in p.index:
        row = p.loc[code_dep]
        props.update(citiz=int(row["Citiz"]), pro=int(row["total_pro"]), getaround=int(row["Getaround"]),
                     part=None if pd.isna(row["part_citiz_pro_%"]) else round(float(row["part_citiz_pro_%"]), 1),
                     statut=row["statut"])
    else:
        props.update(citiz=0, pro=0, getaround=0, part=None, statut="Aucune offre visible")
    light["features"].append({"type": "Feature", "properties": props,
                              "geometry": shapely.geometry.mapping(geom.simplify(0.01))})

STATUT_COLORS = {"Citiz seul": "#00843D", "Citiz leader": "#57B97A", "Citiz challengé": "#B7E4C7",
                 "Absent, concurrence pro": "#F77F00", "Absent, Getaround seul": "#CDB4DB",
                 "Aucune offre visible": "#EEEEEE"}

m2 = folium.Map(location=[46.6, 2.4], zoom_start=6, tiles=None)
folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}", attr="Tiles © Esri — Esri, DeLorme, NAVTEQ", name="Fond clair (Esri)").add_to(m2)
folium.GeoJson(
    light,
    style_function=lambda f: {"fillColor": STATUT_COLORS[f["properties"]["statut"]],
                              "color": "#666", "weight": 0.5, "fillOpacity": 0.75},
    tooltip=folium.GeoJsonTooltip(fields=["nom", "statut", "citiz", "pro", "part", "getaround"],
                                  aliases=["Département", "Statut", "Véhicules Citiz", "Véhicules pro (tous)",
                                           "Part Citiz pro (%)", "Véhicules Getaround"]),
).add_to(m2)
legend = "".join(f'<div><span style="display:inline-block;width:12px;height:12px;background:{c};'
                 f'margin-right:6px;border:1px solid #999"></span>{s}</div>' for s, c in STATUT_COLORS.items())
m2.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;'
    f'border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3);font:13px sans-serif">{legend}</div>'))
m2.save(str(OUT_DIR / "carte_parts_de_marche_departements.html"))
m2

## 7. Pression concurrentielle autour de chaque station Citiz

Pour chaque station Citiz, on compte les véhicules concurrents **à moins de 500 m**, soit environ 5 à 6 minutes à pied, l'ordre de grandeur demandé par le sujet. Une station entourée de beaucoup de concurrents n'est pas forcément un problème : cela peut indiquer une **zone de forte demande**. À croiser plus tard avec la population et l'usage.

In [ ]:
def haversine_counts(src, dst, radius_m=500, chunk=500):
    # Nombre de points de dst à moins de radius_m de chaque point de src (calcul vectorisé par blocs)
    lat2, lon2 = np.radians(dst["lat"].values), np.radians(dst["lon"].values)
    w2 = dst["n_vehicles"].values
    out = np.zeros(len(src))
    for i in range(0, len(src), chunk):
        lat1 = np.radians(src["lat"].values[i:i+chunk])[:, None]
        lon1 = np.radians(src["lon"].values[i:i+chunk])[:, None]
        a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
        d = 2 * 6_371_000 * np.arcsin(np.sqrt(a))
        out[i:i+chunk] = ((d <= radius_m) * w2).sum(axis=1)
    return out

cz = all_points[all_points["operator"] == "Citiz"].copy()
cz["concurrents_pro_500m"] = haversine_counts(cz, all_points[all_points["operator"].isin(["Clem'", "Optymo", "Leo&Go"])])
cz["getaround_500m"] = haversine_counts(cz, all_points[all_points["operator"] == "Getaround"])
cz["autres_citiz_500m"] = haversine_counts(cz, cz) - cz["n_vehicles"]  # véhicules Citiz voisins

print(cz[["concurrents_pro_500m", "getaround_500m", "autres_citiz_500m"]].describe().round(1).to_string())
print(f"\n{(cz['concurrents_pro_500m'] > 0).mean():.0%} des stations Citiz ont un concurrent pro à moins de 500 m, "
      f"{(cz['getaround_500m'] > 0).mean():.0%} ont au moins un Getaround")
cz.sort_values("getaround_500m", ascending=False)[
    ["network", "name", "dep_nom", "n_vehicles", "concurrents_pro_500m", "getaround_500m"]].head(15)

In [ ]:
cz.to_csv(OUT_DIR / f"citiz_stations_pression_concurrentielle_{today}.csv", index=False)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(cz["getaround_500m"].clip(upper=100), bins=50, color=COLORS["Getaround"], alpha=0.8)
ax.set_xlabel("Véhicules Getaround à moins de 500 m (plafonné à 100)")
ax.set_ylabel("Nombre de stations Citiz")
ax.set_title("Exposition des stations Citiz à Getaround", fontweight="bold")
plt.tight_layout(); plt.show()

## 8. Analyse de la concurrence : Montpellier et Nice

**Question** : dans deux grandes villes jeunes où Citiz n'est pas présent, qui sont les concurrents, quelle est leur taille, que sait-on de leur rentabilité, et reste-t-il des parts de marché ?

⚠️ Les sections 1 à 7 reposent sur les flux GBFS ouverts. Cette section ajoute des **informations issues de la presse et de documents publics** (liens et dates dans le tableau), pour les opérateurs **sans flux ouvert** (Modulauto, Mobilize Share). Ces valeurs sont saisies à la main : elles n'ont pas la même fiabilité que les relevés automatiques.

### 8.1 Fiche des concurrents

In [ ]:
# Chiffres issus de la presse / documents publics (vérifiés en octobre 2026). Non mesurés par nos relevés.
profils = pd.DataFrame([
    dict(operateur="Citiz (réseau)", modele="Boucle, coopérative", zone="France, 280 villes",
         flotte=2560, ca_meur=22.2, rentabilite="Résultat net consolidé non publié ; tête de réseau Strasbourg : +28 k€ en 2024",
         fiabilite="Bonne", date="S1 2024", note="11,1 M€ au S1 2024 (x2 = estimation annuelle), 32 300 utilisateurs actifs"),
    dict(operateur="Modulauto", modele="Boucle, société privée (Communauto au capital)", zone="Montpellier, Nîmes, Sète, Béziers, Narbonne, Perpignan…",
         flotte=110, ca_meur=1.7, rentabilite="« Équilibré depuis 2013 » (déclaratif)",
         fiabilite="Moyenne", date="juin 2024", note="Objectif 200+ voitures fin 2024 ; 3 000 conducteurs ; 200+ contrats entreprises/collectivités"),
    dict(operateur="Getaround", modele="Entre particuliers + flotte connectée", zone="France entière, 3 700 flux",
         flotte=None, ca_meur=22.6, rentabilite="Europe « rentable depuis début 2026 » ; France non publiée",
         fiabilite="Moyenne", date="2023 (CA Europe)", note="Vendu à GoMore en mai 2026. Nice : 200+ voitures, occupation 60-76 %, 10 locations/mois/véhicule"),
    dict(operateur="Mobilize Share (Renault)", modele="Boucle, 100 % électrique", zone="Nice et 5 communes voisines",
         flotte=65, ca_meur=None, rentabilite="Service arrêté le 30/06/2026 (recentrage de Mobilize)",
         fiabilite="Bonne (arrêt) / nulle (chiffres)", date="2026", note="38 stations ; la Métropole a repris les 86 bornes (Izivia)"),
    dict(operateur="Clem'", modele="Stations, électrique", zone="France (404 stations dans nos relevés)",
         flotte=int(all_points[all_points.operator == "Clem'"].n_vehicles.sum()), ca_meur=None, rentabilite="Non trouvée",
         fiabilite="Relevé GBFS", date="aujourd'hui", note="Aucune station à Montpellier ni à Nice"),
    dict(operateur="Ubeeqo (Europcar On Demand)", modele="Boucle, B2B + B2C", zone="Grandes villes",
         flotte=None, ca_meur=None, rentabilite="Non trouvée (intégré au groupe Europcar)",
         fiabilite="—", date="—", note="Aucun chiffre isolable"),
])
profils["ca_par_vehicule_k_eur"] = (profils["ca_meur"] * 1000 / profils["flotte"]).where(
    profils["operateur"].isin(["Citiz (réseau)", "Modulauto"]))
profils.set_index("operateur")[["modele", "flotte", "ca_meur", "ca_par_vehicule_k_eur", "rentabilite", "fiabilite", "date"]]

**Lecture des ratios** : le chiffre d'affaires par véhicule n'est calculé que pour Citiz et Modulauto, les deux seuls opérateurs en boucle pour lesquels on a à la fois un CA et une flotte. Les deux valeurs ne sont **pas strictement comparables** : Citiz = doublement du 1er semestre ; Modulauto = prévision 2024 divisée par une flotte mesurée à une autre date, avec une grosse part de contrats entreprises. À présenter comme un **ordre de grandeur (8 à 15 k€ par voiture et par an)**, pas comme un résultat.

### 8.2 Ce que montrent nos relevés autour de Montpellier et de Nice

In [ ]:
VILLES = {  # emprise en degrés (lat_min, lat_max, lon_min, lon_max) ≈ commune + proche couronne
    "Montpellier": (43.55, 43.67, 3.78, 3.97),
    "Nice": (43.64, 43.76, 7.17, 7.33),
    "Toulouse (réf. : Citiz présent)": (43.52, 43.70, 1.33, 1.52),
}
def dans(df, b):
    return df[df.lat.between(b[0], b[1]) & df.lon.between(b[2], b[3])]

rows = []
for ville, b in VILLES.items():
    d = dans(all_points, b)
    rows.append({"ville": ville, **{op: int(d[d.operator == op].n_vehicles.sum()) for op in COLORS}})
releves = pd.DataFrame(rows).set_index("ville")

# Offre connue mais invisible en GBFS (saisie manuelle, cf. 8.1)
releves["Modulauto (hors GBFS)"] = [110, 0, 0]    # toutes villes d'Occitanie confondues : ordre de grandeur pour Montpellier
releves["Mobilize Share (arrêté 06/2026)"] = [0, 65, 0]
releves

In [ ]:
# Couverture de la collecte Getaround : les chiffres ci-dessus sont des PLANCHERS tant que la collecte est partielle
try:
    gdir = sorted(d for d in DATA_DIR.glob("getaround_20*") if d.is_dir())[-1]
    n_feeds = len([p for p in gdir.glob("*.json") if not p.name.startswith("_")])
    n_total = len(json.loads((gdir / "_manifest.json").read_text()))
    print(f"Couverture Getaround : {n_feeds}/{n_total} flux ({n_feeds / n_total:.0%}) — "
          + ("collecte complète" if n_feeds >= n_total else "chiffres Getaround = planchers"))
except Exception as e:
    print("Couverture Getaround inconnue :", e)

In [ ]:
# Cartes Montpellier et Nice : Getaround (violet), Clem'/autres (couleurs habituelles), Citiz (vert) = aucune station
def carte_ville(ville, centre, zoom=12):
    b = VILLES[ville]
    d = dans(all_points, b)
    mv = folium.Map(location=centre, zoom_start=zoom, tiles="CartoDB positron")
    for r in d.itertuples():
        folium.CircleMarker([r.lat, r.lon], radius=3 + 1.5 * math.sqrt(r.n_vehicles), color=COLORS[r.operator],
                            fill=True, fill_opacity=0.6, weight=1,
                            tooltip=f"{r.operator} {r.name}").add_to(mv)
    legend = "".join(f'<div><span style="display:inline-block;width:12px;height:12px;border-radius:6px;'
                     f'background:{c};margin-right:6px"></span>{op}</div>' for op, c in COLORS.items())
    mv.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;'
        f'border-radius:6px;box-shadow:0 1px 4px rgba(0,0,0,.3);font:13px sans-serif">{legend}'
        f'<div style="margin-top:6px;font-size:11px">Modulauto et Mobilize Share : pas de flux GBFS, non affichés</div></div>'))
    mv.save(str(OUT_DIR / f"carte_{ville.lower()}.html"))
    return mv

carte_ville("Montpellier", [43.6108, 3.8767])

In [ ]:
carte_ville("Nice", [43.7102, 7.2620])

### 8.3 Démographie jeune (sources publiques)

| | Montpellier | Nice |
|---|---|---|
| Étudiants | plus de 80 000, soit environ 17 % de la population (unité urbaine, L'Étudiant 2023-2024) | non trouvé de façon fiable |
| 15-29 ans | 25,5 % de la Métropole (INSEE) | non trouvé |

La donnée Nice est **à compléter** (INSEE, recensement ; Université Côte d'Azur). Elle est indispensable avant d'affirmer que Nice est un marché « jeune ».

### 8.4 Matrice d'opportunité

In [ ]:
matrice = pd.DataFrame({
    "Montpellier": {
        "Citiz présent": "Non",
        "Concurrent en boucle": "Modulauto (110 voitures, Communauto au capital)",
        "Concurrent entre particuliers": "Getaround (166 véhicules visibles, plancher)",
        "Dynamique du marché": "Croissance de Modulauto (10-20 %/an visés)",
        "Public jeune": "Fort (80 000+ étudiants)",
        "Fenêtre pour Citiz": "Étroite : entrée face à un acteur local solide, ou partenariat",
        "Priorité": 2,
    },
    "Nice": {
        "Citiz présent": "Non",
        "Concurrent en boucle": "Aucun depuis le 30/06/2026 (Mobilize Share arrêté, 65 voitures)",
        "Concurrent entre particuliers": "Getaround (158 véhicules visibles ; 200+ selon la presse, occupation 60-76 %)",
        "Dynamique du marché": "Demande confirmée, mais 2 opérateurs en boucle ont déjà arrêté depuis 2018",
        "Public jeune": "À documenter",
        "Fenêtre pour Citiz": "Large : plus d'opérateur en station, 86 bornes reprises par la Métropole",
        "Priorité": 1,
    },
})
matrice

### 8.5 Conclusions et réserves

- **Nice est la meilleure cible** : plus aucun opérateur en station et une demande réelle (Getaround y atteint 60 à 76 % d'occupation). **Réserve** : Auto-Bleue (public) puis Renault/Mobilize ont arrêté, ce qui pose la question du **modèle économique** à documenter avant de recommander l'implantation. Un **label autopartage** de la Métropole est requis.
- **Montpellier est un marché tenu** par Modulauto, adossé à Communauto, avec réciprocité d'abonnement. Le scénario réaliste est un **partenariat** ou une niche (collectivités, étudiants dès 18 ans, comme l'exigeait l'appel de la Métropole de mai 2024).
- **Rentabilité** : aucun seuil chiffré en open data. Les deux signaux fiables sont l'équilibre déclaré de Modulauto (avec forte clientèle B2B) et l'arrêt de deux opérateurs à Nice. Pour des comptes réels : Pappers/Infogreffe (SIREN de Modulauto, des SCIC Citiz, de Getaround France).
- **Limite majeure** : les flottes de Modulauto et de Mobilize sont saisies à la main (presse 2023-2024) et **absentes de la carte de la section 5**, qui sous-estime donc la concurrence en Occitanie.

**Sources** : [Modulauto / Communauto](https://communauto.com/modulauto-communauto-accelere-developpement-autopartage) · [Modulauto, Journal des Entreprises](https://www.lejournaldesentreprises.com/article/modulauto-se-structure-pour-developper-lautopartage-2097370) · [Citiz S1 2024](https://www.lejournaldesentreprises.com/article/la-cooperative-dautopartage-citiz-affiche-une-forte-croissance-au-premier-semestre-2024-2104866) · [Getaround à Nice](https://www.tesla-mag.com/getaround-et-la-metropole-nice-cote-dazur-renforcent-leur-partenariat-pour-lautopartage-electrique/) · [Fin de Mobilize Share à Nice](https://www.automobile-propre.com/articles/nice-bornes-recharge-ex-mobilize-share-reseau-public/) · [Getaround vendu à GoMore (CFNews)](https://www.cfnews.net/Acces-libre/Getaround-roule-en-Europe-avec-un-danois-700113) · [L'Étudiant Montpellier](https://www.letudiant.fr/palmares/palmares-des-villes-etudiantes/montpellier.html) · [INSEE Montpellier Métropole](https://www.insee.fr/fr/statistiques/1908394)

## 9. Premiers enseignements (à relire après exécution)

Les chiffres exacts dépendent du moment de la collecte : relancer avec `FORCE_REFRESH = True` pour mettre à jour.

- **Positionnement** : Citiz est un acteur **historique en station**. Sur le périmètre professionnel visible en open data, il est très largement majoritaire. Getaround domine en volume brut, mais sur un autre modèle (particuliers).
- **Géographie** : la carte des statuts fait apparaître les **départements où Citiz est absent alors qu'une demande existe** (Getaround ou un concurrent pro). Ce sont des candidats à l'expansion pour l'objectif 3.
- **Densité locale** : l'analyse à 500 m montre si Citiz est implanté là où l'autopartage entre particuliers est déjà dense. C'est un signal de demande à réutiliser comme variable du modèle.

## 9. Limites (à présenter honnêtement au jury)

1. **Photo instantanée** : free-floating et Getaround ne montrent que les véhicules visibles au moment de la collecte. Un véhicule en cours de location peut manquer.
2. **Couverture incomplète** : les opérateurs sans flux GBFS public sont absents (Communauto, Zity, Free2Move, Ubeeqo, Marcel, MobilizeShare, offres de loueurs…). La part de Citiz est donc **surestimée** sur le périmètre pro, surtout en Île-de-France.
3. **Getaround ≠ concurrent direct** : la plupart des véhicules sont des voitures de particuliers peu disponibles. Les compter à 1 pour 1 avec une Citiz surestime leur poids. C'est pour cela qu'on présente deux périmètres.
4. **Comptage Citiz** : via `vehicle_status` (véhicule rattaché à sa station d'attache). Pour Clem'/Optymo, on prend le maximum entre la capacité et les véhicules disponibles.
5. **Part de marché ≠ usage** : on mesure l'**offre**, pas les locations. Pour l'usage, il faudra un relevé régulier de `station_status` pendant le hackathon.

## 10. Suites

- Rapporter l'offre à la **population** (carroyage INSEE 200 m) : véhicules pour 10 000 habitants par département ou par commune.
- Analyse **par commune / aire urbaine** plutôt que par département.
- Croiser avec les **arrêts de transport en commun** (GTFS), une contrainte d'implantation Citiz.
- Lancer le **relevé périodique** de `station_status` Citiz pour approximer l'usage de chaque station.